# UniswapX Intents and RFQ Lifecycle Analysis

Polaris standardizes RFQs, quotes, executable intents, and settlement observations into one
[Intents and RFQs](https://docs.polaris.supply/schemas/intents-and-rfqs) schema. This notebook
explores live UniswapX observations, contrasts canonical and venue-native fields, follows identifiers
through lifecycle updates, and measures asset-flow and status activity.

Public coverage may contain executable intents without RFQ or quote rows. That is treated as an
observed property of the selected data—not as zero-valued RFQ activity.


## Setup

The example reads the public preview day from catalog metadata and searches its first bounded
windows for data. It caps the iterator at 25,000 observations.


In [ ]:
from itertools import islice

from polaris_data import PolarisClient
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-darkgrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)


def as_utc(value):
    timestamp = pd.Timestamp(value)
    return timestamp.tz_localize("UTC") if timestamp.tzinfo is None else timestamp.tz_convert("UTC")


def accessible_bounds(market_info):
    """Return the no-key catalog interval for an open or preview market."""
    start = as_utc(market_info["start"])
    end = as_utc(market_info["end"])
    access = market_info.get("access") or {}
    cutoff = access.get("public_cutoff_date")
    if access.get("status") == "preview" and cutoff:
        public_day = as_utc(cutoff)
        start = max(start, public_day)
        end = min(end, public_day + pd.Timedelta(days=1))
    if start >= end:
        raise ValueError("Catalog metadata does not expose a no-key interval for this market")
    return start, end


def bounded_rows(iterator, limit):
    """Materialize at most limit rows and close a partially consumed SDK generator."""
    rows = list(islice(iterator, limit + 1))
    truncated = len(rows) > limit
    close = getattr(iterator, "close", None)
    if close is not None:
        close()
    return rows[:limit], truncated


def event_timestamp(row):
    """Support both the legacy and v2 Polaris event envelopes."""
    value = row.get("collector_timestamp", row.get("timestamp"))
    return pd.to_datetime(value, unit="ms", utc=True)


In [ ]:
source = "uniswapx"
market = "intents"
window_length = pd.Timedelta(minutes=20)
max_rows = 25_000


## Discover public intent coverage

Preview datasets expose one catalog-designated UTC day without credentials. The SDK raises for
ranges outside that day, so the notebook intersects catalog coverage with `public_cutoff_date`.


In [ ]:
with PolarisClient() as client:
    catalog = client.catalog(source=source, market=market)

market_info = catalog["markets"][0]
available_start, available_end = accessible_bounds(market_info)
print(f"No-key coverage: {available_start} -> {available_end}")
pd.Series(market_info)


## Fetch the first populated bounded window

Intent sources can have gaps. Up to six consecutive 20-minute windows are checked, and the first
populated one becomes the analysis range.


In [ ]:
intent_rows = []
truncated = False
selected_start = selected_end = None

with PolarisClient() as client:
    for offset in range(6):
        candidate_start = available_start + offset * window_length
        candidate_end = min(candidate_start + window_length, available_end)
        if candidate_start >= candidate_end:
            break
        candidate_rows, candidate_truncated = bounded_rows(
            client.intents(
                source=source, market=market, from_=candidate_start, to=candidate_end, allow_gaps=True,
            ),
            max_rows,
        )
        if candidate_rows:
            intent_rows = candidate_rows
            truncated = candidate_truncated
            selected_start, selected_end = candidate_start, candidate_end
            break

if not intent_rows:
    raise ValueError("No public intent observations were found in the first two accessible hours")

print(f"Selected window: {selected_start} -> {selected_end}")
print(f"Rows materialized: {len(intent_rows):,}; hit cap: {truncated}")


## Canonical shape versus raw venue payload

Canonical fields under `data` are intended for cross-venue analysis. `raw` retains the captured
venue payload when available. Numeric asset amounts remain strings to preserve source precision.


In [ ]:
first = intent_rows[0]
canonical_preview = {
    "envelope": {key: first.get(key) for key in [
        "collector_timestamp", "collector_sequence", "source", "market", "type"
    ]},
    "canonical_data": first.get("data", {}),
    "raw_top_level_keys": sorted((first.get("raw") or {}).keys()),
}
canonical_preview


In [ ]:
records = []
for row in intent_rows:
    data = row.get("data") or {}
    quote = data.get("quote") or {}
    inputs = data.get("inputs") or []
    outputs = data.get("outputs") or []
    correlation_id = data.get("intent_id") or data.get("rfq_id")
    records.append({
        "timestamp": event_timestamp(row),
        "correlation_id": correlation_id,
        "intent_id": data.get("intent_id"),
        "rfq_id": data.get("rfq_id"),
        "quote_id": quote.get("quote_id"),
        "status": data.get("status"),
        "amount_kind": data.get("amount_kind"),
        "input_asset": inputs[0].get("asset_id") if inputs else None,
        "output_asset": outputs[0].get("asset_id") if outputs else None,
        "input_count": len(inputs),
        "output_count": len(outputs),
        "transaction_count": len(data.get("transactions") or []),
        "has_raw": "raw" in row,
    })

intents_df = pd.DataFrame(records).sort_values("timestamp")
field_coverage = (
    intents_df.notna().mean().mul(100).round(1).rename("populated_percent").to_frame()
)
field_coverage


## RFQ, quote, and lifecycle coverage

Rows are observations rather than pre-joined lifecycles. `intent_id` and `rfq_id` are used as
correlation keys; status transitions remain in storage order.


In [ ]:
coverage_summary = pd.DataFrame([{
    "observations": len(intents_df),
    "unique_correlations": intents_df["correlation_id"].nunique(),
    "intent_rows": intents_df["intent_id"].notna().sum(),
    "rfq_rows": intents_df["rfq_id"].notna().sum(),
    "quote_rows": intents_df["quote_id"].notna().sum(),
    "settlement_rows": intents_df["transaction_count"].gt(0).sum(),
}])
display(coverage_summary)

if coverage_summary.loc[0, "rfq_rows"] == 0:
    print("No RFQ observations were published in this public UniswapX sample.")
if coverage_summary.loc[0, "quote_rows"] == 0:
    print("No quote observations were published in this public UniswapX sample.")

lifecycle_summary = (
    intents_df.dropna(subset=["correlation_id"])
    .groupby("correlation_id")
    .agg(
        first_seen=("timestamp", "min"),
        last_seen=("timestamp", "max"),
        observations=("timestamp", "size"),
        statuses=("status", lambda values: " -> ".join(dict.fromkeys(values.dropna().astype(str)))),
        settlement_updates=("transaction_count", lambda values: values.gt(0).sum()),
    )
    .sort_values(["observations", "last_seen"], ascending=False)
)
lifecycle_summary.head(10)


## Status activity and canonical asset flows

Asset identifiers remain canonical CAIP-style strings when chain metadata is available. The charts
intentionally count observations rather than treating incomparable token base units as one volume.


In [ ]:
intents_df["asset_pair"] = (
    intents_df["input_asset"].fillna("unknown").str.rsplit(":", n=1).str[-1].str.slice(0, 12)
    + " → "
    + intents_df["output_asset"].fillna("unknown").str.rsplit(":", n=1).str[-1].str.slice(0, 12)
)
status_counts = intents_df["status"].fillna("unspecified").value_counts()
pair_counts = intents_df["asset_pair"].value_counts().head(10)
observations_per_minute = intents_df.set_index("timestamp").resample("1min").size()

display(pair_counts.rename("observations").to_frame())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
status_counts.sort_values().plot.barh(ax=axes[0], color="#4c78a8")
axes[0].set_title("Lifecycle status observations")
axes[0].set_xlabel("Rows")

observations_per_minute.plot(ax=axes[1], color="#f58518")
axes[1].set_title("Observation rate")
axes[1].set_ylabel("Rows per minute")
axes[1].set_xlabel("Time (UTC)")

pair_counts.sort_values().plot.barh(ax=axes[2], color="#54a24b")
axes[2].set_title("Top canonical asset flows")
axes[2].set_xlabel("Rows")

plt.tight_layout()
plt.show()
